# GF(8) generalized toric codes: evolutionary search on Kaggle

This notebook runs the fixed-dimension evolutionary search from `evolve_search.py` on a Kaggle NVIDIA GPU for `k = 12..19`. It is a candidate factory: results are sampled and marked `verified: false`; promising candidates still need a stronger recheck or Magma/BZ verification.

Kaggle setup:

1. Create a Kaggle Notebook.
2. Set **Accelerator** to a GPU such as T4, P100, or A100.
3. Run the first cell to clone the GitHub repo into `/kaggle/working/generalized-toric-gpu-search`.
4. Run the remaining cells. JSONL result files are written to `/kaggle/working` and can be downloaded from the notebook output.

The notebook output is intentionally compact: one progress stream per `k`, then a summary table with the best saved set for each dimension. Full candidate records stay in the JSONL files.

Note: this repository is currently GF(8), length 49. The often-mentioned `[36,19,12]` example is GF(7), length 36, and would need a separate GF(7) evaluation matrix and kernels.

In [ ]:
%cd /kaggle/working
!rm -rf generalized-toric-gpu-search
!git clone https://github.com/t-ravnushkin/generalized-toric-gpu-search.git

import os
import sys

REPO = "/kaggle/working/generalized-toric-gpu-search"
sys.path = [p for p in sys.path if "generalized-toric-gpu-search" not in p]
sys.path.insert(0, REPO)
os.chdir("/kaggle/working")

# If this cell is rerun without restarting the kernel, discard stale imports.
for name in [
    "evolve_search", "kernel_cuda_bp", "kernel", "precompute",
    "codetables", "gf8",
]:
    sys.modules.pop(name, None)

print(f"Using repo: {REPO}")


In [ ]:
# GPU sanity check. CuPy is normally preinstalled on Kaggle GPU images.
REQUIRE_CUDA = True

try:
    import cupy as cp
    n_gpu = cp.cuda.runtime.getDeviceCount()
    print(f"CuPy {cp.__version__}; CUDA devices: {n_gpu}")
    for i in range(n_gpu):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(
            f"Device {i}: {props['name'].decode()} "
            f"({props.get('multiProcessorCount', 0)} SMs, "
            f"{props['totalGlobalMem'] // 1024**3} GB VRAM)"
        )
except Exception as exc:
    if REQUIRE_CUDA:
        raise RuntimeError("Enable a Kaggle GPU accelerator before running the search.") from exc
    print(f"CUDA unavailable; evolve_search.py will fall back if possible: {exc}")

In [ ]:
# Choose the search range and runtime profile.
from codetables import bounds_for_n

K_VALUES = list(range(12, 20))
FALLBACK_TARGETS = {12: 28, 13: 27, 14: 26, 15: 24, 16: 24, 17: 23, 18: 21, 19: 21}

try:
    bounds = bounds_for_n(49, q=8, cache=True)
    TARGETS = {k: bounds[k] for k in K_VALUES}
except Exception as exc:
    print(f"Could not fetch codetables bounds; using bundled fallback: {exc}")
    TARGETS = {k: FALLBACK_TARGETS[k] for k in K_VALUES}

# Start with quick to test that the notebook and GPU are healthy.
# Switch to serious for an overnight-style run.
PRESET = "quick"  # "quick" or "serious"

# Sampled scores guide the population. Complete recheck is the only thing
# reported as an actual distance in the final table.
COMPLETE_RECHECK = True

if PRESET == "quick":
    GENERATIONS = 20
    POPULATION_SIZE = 300
    SAMPLE_COUNT = 50_000
    RECHECK_SAMPLE_COUNT = 0
    RECHECK_ROUNDS = 1
    RECHECK_TOP = 2      # exact-test only the final top few; complete is expensive
    LOG_EVERY = 10
elif PRESET == "serious":
    GENERATIONS = 200
    POPULATION_SIZE = 300
    SAMPLE_COUNT = 200_000
    RECHECK_SAMPLE_COUNT = 0
    RECHECK_ROUNDS = 1
    RECHECK_TOP = 4      # raise cautiously; exact k>=12 checks can be slow
    LOG_EVERY = 20
else:
    raise ValueError("PRESET must be 'quick' or 'serious'")

ELITE_COUNT = 30
PARENT_POOL = 200
MUTATION_RATE = 0.10
RECHECK_EVERY = GENERATIONS  # complete recheck only at the end of each k run
SEED_BASE = 1
BATCH_SIZE = None  # None lets evolve_search choose from the GPU SM count.
SEED_SETS_BY_K = {}  # Optional: {k: [(indices...), ...]} for seeded restarts.

print(f"Preset: {PRESET}")
print("Targets:")
for k in K_VALUES:
    print(f"  GF(8) [49,{k},?>={TARGETS[k]}]")


In [ ]:
# Run the k=12..19 sweep. Candidates are saved to JSONL files in /kaggle/working.
from datetime import datetime
from pathlib import Path
from evolve_search import init_oracles, run_evolution

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
ORACLES_INFO = init_oracles()  # Compile/init CUDA once and reuse it for every k.
RESULT_FILES = {}

for k in K_VALUES:
    results_file = Path(f"/kaggle/working/evolve_k{k}_{PRESET}_{ts}.json")
    RESULT_FILES[k] = results_file
    print("\n" + "=" * 72)
    print(f"k={k} target_d={TARGETS[k]} -> {results_file}")
    print("=" * 72)
    run_evolution(
        k=k,
        target_d=TARGETS[k],
        generations=GENERATIONS,
        population_size=POPULATION_SIZE,
        elite_count=ELITE_COUNT,
        parent_pool=PARENT_POOL,
        mutation_rate=MUTATION_RATE,
        sample_count=SAMPLE_COUNT,
        recheck_sample_count=RECHECK_SAMPLE_COUNT,
        recheck_rounds=RECHECK_ROUNDS,
        recheck_top=RECHECK_TOP,
        recheck_every=RECHECK_EVERY,
        batch_size=BATCH_SIZE,
        seed=SEED_BASE + 10_000 * k,
        seed_sets=[tuple(s) for s in SEED_SETS_BY_K.get(k, [])],
        results_file=results_file,
        oracles_info=ORACLES_INFO,
        log_every=LOG_EVERY,
        complete_recheck=COMPLETE_RECHECK,
    )

print("\nSweep complete.")
for k, path in RESULT_FILES.items():
    print(f"k={k}: {path}")


In [ ]:
# Compact actual results: complete-kernel verified candidates only.
import json
from IPython.display import FileLink, display

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def point_str(points):
    return "[" + ", ".join(f"({a},{b})" for a, b in points) + "]"

SUMMARY = []
for k in K_VALUES:
    path = RESULT_FILES[k]
    records = load_jsonl(path)
    verified = [
        r for r in records
        if r.get("type") == "candidate"
        and r.get("verified") is True
        and "min_distance" in r
    ]
    complete = next((r for r in reversed(records) if r.get("type") == "run_complete"), {})
    if verified:
        best = max(
            verified,
            key=lambda r: (r.get("min_distance", -1), r.get("sampled_min_distance", -1)),
        )
        SUMMARY.append({
            "k": k,
            "target": TARGETS[k],
            "status": "verified",
            "actual_d": best.get("min_distance"),
            "max_zeros": best.get("max_zeros"),
            "sampled_d": best.get("sampled_min_distance"),
            "indices": best.get("indices"),
            "points": best.get("lattice_points"),
            "file": path,
        })
    else:
        SUMMARY.append({
            "k": k,
            "target": TARGETS[k],
            "status": "none",
            "actual_d": None,
            "max_zeros": None,
            "sampled_d": complete.get("best_sampled_min_distance"),
            "indices": None,
            "points": None,
            "file": path,
        })

print("Complete-kernel verified results")
print("k  target  status    actual_d  max_zeros  sampled_best")
print("-" * 64)
for row in SUMMARY:
    actual = "-" if row["actual_d"] is None else str(row["actual_d"])
    mz = "-" if row["max_zeros"] is None else str(row["max_zeros"])
    sampled = "-" if row["sampled_d"] is None else str(row["sampled_d"])
    print(
        f"{row['k']:2d} {row['target']:7d} {row['status']:<9s} "
        f"{actual:>8s} {mz:>10s} {sampled:>13s}"
    )

print("\nVerified sets")
for row in SUMMARY:
    if row["actual_d"] is None:
        print(f"\nk={row['k']}: no complete-kernel verified candidate in this run")
        continue
    print(f"\nk={row['k']} d={row['actual_d']} file={row['file'].name}")
    print(f"indices: {row['indices']}")
    print(f"points : {point_str(row['points'])}")

print("\nDownloads")
for row in SUMMARY:
    display(FileLink(str(row["file"])))


## What to do with results

The table above reports only complete-kernel verified candidates. For those rows, `actual_d` is the actual minimum distance returned by the complete evaluation, because passing the target check means the kernel scanned every nonzero codeword without early abort.

Rows marked `none` mean the sampled search proposed candidates, but none of the final top candidates passed the complete target check. Increase `GENERATIONS`, `SAMPLE_COUNT`, or `RECHECK_TOP` if you want to spend more GPU time on exact verification.